# Aasaan Khata (FYP) - Qwen2.5-VL-7B QLoRA Fine-Tuning
**Hardware:** Kaggle T4 x 2 or P100 (16GB GPU) | **Team:** Irbaz, Talal, Ibrahim
Fine-tuning Qwen2.5-VL-7B end-to-end on handwritten Urdu (Nastaliq) wholesale ledgers.

In [ ]:
# 1. Install dependencies
!pip install -q -U transformers accelerate peft bitsandbytes qwen-vl-utils wandb


In [ ]:
# 2. Authenticate Hugging Face & W&B (Optional)
from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")
import wandb
# wandb.login(key="YOUR_WANDB_API_KEY")


In [ ]:
# 3. Model & Quantization Setup
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "Qwen/Qwen2.5-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading base model in 4-bit...")
processor = AutoProcessor.from_pretrained(model_id)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
)

model = prepare_model_for_kbit_training(model)
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# 4. Dataset Loader for Qwen Conversational Schema
import os, json
from PIL import Image
from torch.utils.data import Dataset
from qwen_vl_utils import process_vision_info

class KaggleLedgerDataset(Dataset):
    def __init__(self, json_path, processor, max_pixels=602112):
        self.processor = processor
        self.max_pixels = max_pixels
        with open(json_path, "r", encoding="utf-8") as f:
            self.data = json.load(f)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = Image.open(item["image"]).convert("RGB")
        conv = item["conversations"]
        user_text = conv[0]["content"].replace("<image>
", "").strip()
        assistant_text = conv[1]["content"]

        messages = [
            {"role": "user", "content": [{"type": "image", "image": image, "max_pixels": self.max_pixels}, {"type": "text", "text": user_text}]},
            {"role": "assistant", "content": assistant_text}
        ]
        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = self.processor(text=[text], images=image_inputs, videos=video_inputs, padding=False, return_tensors="pt")

        input_ids = inputs["input_ids"][0]
        labels = input_ids.clone()
        marker = "<|im_start|>assistant
"
        enc_marker = self.processor.tokenizer.encode(marker, add_special_tokens=False)
        m_len = len(enc_marker)
        for i in range(len(input_ids) - m_len):
            if input_ids[i:i+m_len].tolist() == enc_marker:
                labels[:i+m_len] = -100
                break

        return {
            "input_ids": input_ids,
            "labels": labels,
            "pixel_values": inputs.get("pixel_values", [None])[0],
            "image_grid_thw": inputs.get("image_grid_thw", [None])[0]
        }


In [ ]:
# 5. Trainer & Hyperparameters
from transformers import TrainingArguments, Trainer

def collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]
    labels = [item["labels"] for item in batch]
    input_ids_padded = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=151643)
    labels_padded = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)
    batch_out = {"input_ids": input_ids_padded, "labels": labels_padded, "attention_mask": input_ids_padded.ne(151643).long()}
    if batch[0]["pixel_values"] is not None:
        batch_out["pixel_values"] = torch.cat([b["pixel_values"] for b in batch], dim=0)
    if batch[0]["image_grid_thw"] is not None:
        batch_out["image_grid_thw"] = torch.cat([b["image_grid_thw"] for b in batch], dim=0)
    return batch_out

training_args = TrainingArguments(
    output_dir="./qwen_ledger_adapter",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=5,
    save_strategy="epoch",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none"
)

train_dataset = KaggleLedgerDataset("dataset/train.json", processor)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn
)
trainer.train()
model.save_pretrained("./qwen_ledger_adapter")
print("Adapter successfully saved!")
